# WISE Infrared Photometry: Acquisition, Preprocessing & Cross-matching
### Companion to `sdss_spectra_clustering.ipynb` — reproducing Jespersen et al. (2025)
### *"The optical and infrared are connected"* ([arXiv:2503.03816](https://arxiv.org/abs/2503.03816))

---

**What this notebook does:**

1. Queries the SDSS SkyServer for WISE photometry cross-matched to SDSS spectroscopic galaxies
2. Applies quality cuts from §2.2 and Appendix C of the paper
3. Converts Vega magnitudes to flux densities (Jy) using Wright et al. (2010) zero-points
4. Saves a clean `.npz` archive ready for MLP training / PP-plot evaluation

**Two operating modes:**

| Mode | Trigger | Best for |
|------|---------|----------|
| `astroquery` | `DATA_SOURCE = "astroquery"` | Prototyping (≤10k objects) |
| CSV file     | `DATA_SOURCE = "csv"` | Full scale (510k, from CasJobs download) |

**How the paper gets WISE data (§2.2):**
- Downloaded from the `wise_allsky` table on the SDSS SkyServer
- SDSS-WISE cross-match from the `wise_xmatch` table (astrometric, max 2", requires W1 & W2 detection)
- Uses **original Vega magnitudes** (not AB) for W1–W4
- Zero-point flux densities from Wright et al. (2010) for Jansky conversion

---

### Quick physics context

WISE mapped the entire sky in four infrared bands. These probe different physics than optical spectra:

| Band | Wavelength | Physical origin |
|------|-----------|----------------|
| W1   | 3.4 µm    | Old stellar populations |
| W2   | 4.6 µm    | Old stars + hot AGN dust |
| W3   | 12.1 µm   | PAH emission + warm dust |
| W4   | 22.2 µm   | Hot dust, star formation |

The paper's claim: **all of this IR information is already encoded in the optical SDSS spectrum**.

## Cell 0 — Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time, sys, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("WISE")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

print(f"Python {sys.version.split()[0]}  |  NumPy {np.__version__}  |  Pandas {pd.__version__}")
print("All imports OK ✓")

## Cell 1 — Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SCALE KNOBS
# ═══════════════════════════════════════════════════════════════════════
N_SPECTRA   = None            # None → all (~510k). Set a number for quick tests.
DATA_SOURCE = "csv"           # "csv"        ← pre-downloaded by data_download.ipynb (recommended)
                               # "astroquery" ← live query (≤ 10k objects only)

# ── Path for DATA_SOURCE = "csv" ───────────────────────────────────────
# data_download.ipynb saves WISE photometry here:
CSV_PATH = Path("wise_raw/wise_crossmatch.csv")
# If you still have an old CasJobs-downloaded file, point to it instead:
# CSV_PATH = Path("wise_output/wise_crossmatch_casjobs.csv")
# ═══════════════════════════════════════════════════════════════════════

Z_MAX = 0.5

# WISE zero-point flux densities in Jansky (Wright et al. 2010, Table 1)
# F_Jy = F0 * 10^(-m / 2.5)
WISE_F0_JY = {"W1": 309.540, "W2": 171.787, "W3": 31.674, "W4": 8.363}
WISE_WAVELENGTHS_UM = {"W1": 3.4, "W2": 4.6, "W3": 12.1, "W4": 22.2}

OUT_DIR  = Path("wise_output");  OUT_DIR.mkdir(exist_ok=True)
SDSS_DIR = Path("sdss_output")

log.info(f"Config: N_SPECTRA={N_SPECTRA}, DATA_SOURCE='{DATA_SOURCE}', CSV_PATH={CSV_PATH}")


## Cell 2 — SQL query builder

This three-table JOIN reproduces how the paper obtained its data:
```
SpecObj → PhotoObj (bestobjid=objid) → wise_xmatch (sdss_objid=objid) → wise_allsky (wise_cntr=cntr)
```
**Important:** `wise_xmatch` joins on `sdss_objid` to `PhotoObj.objid`
(it's a photometric cross-match). To get spectroscopic galaxies,
we first join `SpecObj → PhotoObj`, then into WISE.

**Copy the printed SQL into CasJobs for the full dataset.**

In [ ]:
def build_wise_sql(n=None, z_max=0.5):
    """
    Build SQL for WISE-crossmatched SDSS galaxy photometry.
    
    KEY DETAIL: wise_xmatch joins on sdss_objID -> PhotoObj.objid
    (it's a photometric cross-match, not spectroscopic).
    So the chain is: SpecObj -> PhotoObj -> wise_xmatch -> wise_allsky.
    """
    top = f"TOP {n}" if n is not None else ""
    return f"""
SELECT {top}
    s.specobjid, s.plate, s.mjd, s.fiberid,
    s.z AS redshift, s.class AS specclass, s.subclass AS specsubclass,
    p.petroMag_r,
    w.w1mpro, w.w2mpro, w.w3mpro, w.w4mpro,
    w.w1sigmpro, w.w2sigmpro, w.w3sigmpro, w.w4sigmpro,
    w.w1snr, w.w2snr, w.w3snr, w.w4snr
FROM SpecObj AS s
JOIN PhotoObj AS p ON s.bestobjid = p.objid
JOIN wise_xmatch AS x ON x.sdss_objid = p.objid
JOIN wise_allsky AS w ON x.wise_cntr = w.cntr
WHERE s.class = 'GALAXY'
  AND s.z BETWEEN 0.01 AND {z_max}
  AND s.zwarning = 0
  AND p.petroMag_r BETWEEN 14.0 AND 17.8
  AND p.petroMag_r != -9999
ORDER BY s.plate, s.mjd, s.fiberid
""".strip()


print("SQL for CasJobs (paste this for the full dataset):")
print("=" * 60)
print(build_wise_sql(n=None, z_max=Z_MAX))
print("=" * 60)

## Cell 3 — Load data (fully implemented: astroquery *or* CSV)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# DIAGNOSTIC: Raw query test — see exactly what the server returns
# Run this BEFORE the main data-loading cell to debug query issues.
# ═══════════════════════════════════════════════════════════════════════
import requests

# Try a tiny test query first to verify the tables exist
test_queries = [
    ("Basic SpecObj",
     "SELECT TOP 2 specobjid FROM SpecObj WHERE class='GALAXY'"),
    ("wise_xmatch exists?",
     "SELECT TOP 2 sdss_objid, wise_cntr, dist FROM wise_xmatch"),
    ("wise_allsky exists?",
     "SELECT TOP 2 cntr, w1mpro FROM wise_allsky"),
    ("Full JOIN (2 rows)",
     build_wise_sql(n=2, z_max=Z_MAX)),
]

# astroquery uses this endpoint under the hood:
SKYSERVER_URL = "https://skyserver.sdss.org/dr18/SkyServerWS/SearchTools/SqlSearch"

for label, sql in test_queries:
    print(f"\n{'─'*60}")
    print(f"TEST: {label}")
    print(f"SQL:  {sql[:120]}{'...' if len(sql)>120 else ''}")
    try:
        resp = requests.get(
            SKYSERVER_URL,
            params={"cmd": sql, "format": "csv"},
            timeout=30,
        )
        # Show first 500 chars of response (will reveal error messages)
        text = resp.text[:500]
        if resp.status_code != 200:
            print(f"  ⚠️  HTTP {resp.status_code}")
        elif 'error' in text.lower() or '<html' in text.lower():
            print(f"  ⚠️  Server returned error page:")
            print(f"  {text[:300]}")
        else:
            lines = text.strip().split('\n')
            print(f"  ✓ Got {len(lines)} lines of CSV")
            for line in lines[:4]:
                print(f"    {line[:120]}")
    except Exception as e:
        print(f"  ✗ {type(e).__name__}: {e}")

print(f"\n{'─'*60}")
print("If a test above shows an error, that pinpoints the problem.")
print("Common fixes:")
print("  - Wrong DR: try changing dr18 → dr16 or dr17 in the URL above")
print("  - Table missing: wise_xmatch may not exist in all DRs")
print("  - Column name wrong: check Schema Browser for exact names")

In [ ]:
def load_via_astroquery(n, z_max):
    """Live query to SDSS SkyServer. Good for prototyping."""
    from astroquery.sdss import SDSS
    sql = build_wise_sql(n=n, z_max=z_max)
    log.info(f"Sending astroquery SQL (TOP {n}) ...")
    
    # Try with explicit data_release — wise tables available DR10+
    # astroquery's default DR may not have them.
    for dr in [18, 17, 16]:
        log.info(f"Trying DR{dr} ...")
        try:
            result = SDSS.query_sql(sql, timeout=300, data_release=dr)
            if result is not None and len(result) > 0:
                log.info(f"✓ DR{dr} worked! Got {len(result)} rows")
                return result.to_pandas()
            else:
                log.warning(f"DR{dr} returned empty result")
        except Exception as e:
            log.warning(f"DR{dr} failed: {type(e).__name__}: {e}")
            continue
    
    # If astroquery fails on all DRs, fall back to direct HTTP
    log.info("astroquery failed on all DRs. Trying direct HTTP ...")
    import requests, io
    for dr in [18, 17, 16]:
        url = f"https://skyserver.sdss.org/dr{dr}/SkyServerWS/SearchTools/SqlSearch"
        try:
            resp = requests.get(url, params={"cmd": sql, "format": "csv"}, timeout=300)
            text = resp.text.strip()
            if 'error' in text[:200].lower() or '<html' in text[:200].lower():
                log.warning(f"DR{dr} HTTP returned error: {text[:200]}")
                continue
            # Parse CSV
            df = pd.read_csv(io.StringIO(text))
            if len(df) > 0:
                log.info(f"✓ Direct HTTP DR{dr} worked! Got {len(df)} rows")
                return df
        except Exception as e:
            log.warning(f"DR{dr} HTTP failed: {type(e).__name__}: {e}")
            continue
    
    raise RuntimeError(
        "All query methods failed. Options:\n"
        "  1) Run the diagnostic cell above to see the server error\n"
        "  2) Use CasJobs (https://skyserver.sdss.org/casjobs/) to download CSV\n"
        "  3) Set DATA_SOURCE='csv' and point to the downloaded file"
    )


def load_via_csv(csv_path, n=None):
    """
    Load pre-downloaded CasJobs CSV. Works offline, any size.
    The full 510k dataset is ~80-120 MB and loads in <2s.
    """
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\n"
            f"Download it from CasJobs (see Cell 2 for SQL)."
        )
    log.info(f"Loading {csv_path} ({csv_path.stat().st_size/1e6:.1f} MB)")
    t0 = time.time()
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.lower().str.strip()
    log.info(f"Loaded {len(df):,} rows in {time.time()-t0:.1f}s")
    if n is not None and len(df) > n:
        log.info(f"Subsampling to {n} rows")
        df = df.head(n)
    return df


if DATA_SOURCE == "astroquery":
    df_raw = load_via_astroquery(N_SPECTRA, Z_MAX)
elif DATA_SOURCE == "csv":
    df_raw = load_via_csv(CSV_PATH, n=N_SPECTRA)
else:
    raise ValueError(f"Unknown DATA_SOURCE='{DATA_SOURCE}'")

# Normalise column names
df_raw.columns = df_raw.columns.str.lower().str.strip()
print(f"\nRaw data: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

## Cell 4 — Diagnostic: raw data peek

In [ ]:
print("═" * 70)
print("RAW DATA PEEK")
print("═" * 70)

# Check required columns
required = ["specobjid","redshift","petromag_r",
            "w1mpro","w2mpro","w3mpro","w4mpro",
            "w1sigmpro","w2sigmpro","w3sigmpro","w4sigmpro",
            "w1snr","w2snr","w3snr","w4snr"]
missing = [c for c in required if c not in df_raw.columns]
if missing:
    print(f"\n⚠️  MISSING COLUMNS: {missing}")
    print(f"   Available: {sorted(df_raw.columns.tolist())}")
else:
    print(f"✓ All {len(required)} required columns present")

# Null summary
print(f"\nNull counts:")
for col in required:
    if col in df_raw.columns:
        n_null = df_raw[col].isnull().sum()
        flag = " ⚠️" if n_null > 0 else ""
        print(f"  {col:<15s}: {n_null:>7,}{flag}")

# Duplicates
n_dup = df_raw["specobjid"].duplicated().sum()
print(f"\nDuplicate specobjids: {n_dup}")

# Quick dtype check
print(f"\nFirst 2 rows:")
print(df_raw.head(2).to_string())

## Cell 5 — Preprocess WISE photometry

In [ ]:
def preprocess_wise(df, snr_threshold=2.0):
    """Clean WISE photometry into arrays for MLP training."""
    df = df.drop_duplicates(subset="specobjid", keep="first").reset_index(drop=True)
    N = len(df)
    log.info(f"Preprocessing {N:,} galaxies")
    
    mag_cols = ["w1mpro","w2mpro","w3mpro","w4mpro"]
    err_cols = ["w1sigmpro","w2sigmpro","w3sigmpro","w4sigmpro"]
    snr_cols = ["w1snr","w2snr","w3snr","w4snr"]
    f0_list  = [WISE_F0_JY[b] for b in ["W1","W2","W3","W4"]]
    
    mag      = np.full((N,4), np.nan, dtype=np.float32)
    mag_err  = np.full((N,4), 1e4,   dtype=np.float32)  # paper's convention for non-det
    flux_jy  = np.full((N,4), np.nan, dtype=np.float32)
    detected = np.zeros((N,4), dtype=bool)
    
    issues = {"nan_mag":0, "neg_err":0, "low_snr":0, "bad_range":0}
    
    for j,(mc,ec,sc,f0) in enumerate(zip(mag_cols,err_cols,snr_cols,f0_list)):
        m   = pd.to_numeric(df[mc], errors="coerce").values
        sig = pd.to_numeric(df[ec], errors="coerce").values
        snr = pd.to_numeric(df[sc], errors="coerce").values
        
        ok_fin  = np.isfinite(m) & np.isfinite(sig) & np.isfinite(snr)
        ok_err  = sig > 0
        ok_snr  = snr >= snr_threshold
        ok_rng  = (m > 0) & (m < 25)
        good    = ok_fin & ok_err & ok_snr & ok_rng
        
        issues["nan_mag"]  += int((~ok_fin).sum())
        issues["neg_err"]  += int((ok_fin & ~ok_err).sum())
        issues["low_snr"]  += int((ok_fin & ok_err & ~ok_snr).sum())
        issues["bad_range"] += int((ok_fin & ok_err & ok_snr & ~ok_rng).sum())
        
        mag[good,j]      = m[good].astype(np.float32)
        mag_err[good,j]  = sig[good].astype(np.float32)
        flux_jy[good,j]  = (f0 * 10**(-m[good]/2.5)).astype(np.float32)
        detected[good,j] = True
    
    total_bad = sum(issues.values())
    log.info(f"Flagged {total_bad}/{N*4} band-measurements as non-detections")
    for reason, count in issues.items():
        if count > 0:
            log.info(f"  {reason}: {count:,}")
    
    return {
        "specobjid":   df["specobjid"].values.astype(np.int64),
        "redshift":    pd.to_numeric(df["redshift"],  errors="coerce").values.astype(np.float32),
        "petromag_r":  pd.to_numeric(df["petromag_r"], errors="coerce").values.astype(np.float32),
        "mag":mag, "mag_err":mag_err, "flux_jy":flux_jy, "detected":detected,
    }


wise_data = preprocess_wise(df_raw)

## Cell 6 — Diagnostic: preprocessed data summary

In [ ]:
N_gal = len(wise_data["specobjid"])
print("═" * 70)
print(f"PREPROCESSED WISE DATA  ({N_gal:,} galaxies)")
print("═" * 70)

print(f"\n{'Band':<6} {'N_det':>8} {'%det':>6}  {'min':>7} {'median':>7} {'max':>7}  {'σ_med':>7}")
print("-" * 62)
for j, band in enumerate(["W1","W2","W3","W4"]):
    det = wise_data["detected"][:,j]
    n_det = det.sum()
    pct = 100*n_det/N_gal
    if n_det > 0:
        m = wise_data["mag"][det,j]
        e = wise_data["mag_err"][det,j]
        print(f"{band:<6} {n_det:>8,} {pct:>5.1f}%  {m.min():>7.2f} {np.median(m):>7.2f} {m.max():>7.2f}  {np.median(e):>7.4f}")
    else:
        print(f"{band:<6} {n_det:>8,} {pct:>5.1f}%  {'—':>7} {'—':>7} {'—':>7}  {'—':>7}")

# Spot-check
print(f"\nSpot check (first 3):")
for i in range(min(3, N_gal)):
    z = wise_data["redshift"][i]
    m = wise_data["mag"][i]
    d = wise_data["detected"][i]
    bands_str = "  ".join(f"W{j+1}={m[j]:.2f}" if d[j] else f"W{j+1}=—" for j in range(4))
    print(f"  [{i}] z={z:.4f}  {bands_str}")

# Sanity checks
n_nan_z = np.isnan(wise_data["redshift"]).sum()
print(f"\n{'✓' if n_nan_z==0 else '⚠️'} NaN redshifts: {n_nan_z}")
for j, b in enumerate(["W1","W2"]):
    pct = 100*wise_data["detected"][:,j].mean()
    print(f"{'✓' if pct>95 else '⚠️'} {b} detection rate: {pct:.1f}% (expect ~100% from cross-match req.)")

## Cell 7 — Diagnostic plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Row 0: Redshift, Detection fractions, petroMag_r
ax = axes[0,0]
zs = wise_data["redshift"]; zs = zs[np.isfinite(zs)]
ax.hist(zs, bins=40, color="seagreen", edgecolor="white")
ax.set_xlabel("Redshift"); ax.set_ylabel("Count")
ax.set_title("Redshift distribution")

ax = axes[0,1]
bands = ["W1","W2","W3","W4"]
det_pcts = [100*wise_data["detected"][:,j].mean() for j in range(4)]
colors_b = ["#1f77b4","#ff7f0e","#2ca02c","#d62728"]
ax.bar(bands, det_pcts, color=colors_b)
ax.set_ylabel("Detection %"); ax.set_title("Detection rates"); ax.set_ylim(0,105)
for i,v in enumerate(det_pcts): ax.text(i, v+1, f"{v:.0f}%", ha="center", fontsize=9)

ax = axes[0,2]
pmr = wise_data["petromag_r"]; pmr = pmr[np.isfinite(pmr)]
ax.hist(pmr, bins=40, color="mediumpurple", edgecolor="white")
ax.set_xlabel("petroMag_r"); ax.set_ylabel("Count")
ax.set_title("r-band magnitude distribution")

# Row 1: magnitude histograms W1, W2, W3
for j in range(3):
    ax = axes[1,j]
    det = wise_data["detected"][:,j]
    if det.sum() > 0:
        ax.hist(wise_data["mag"][det,j], bins=40, color=colors_b[j], edgecolor="white", alpha=0.8)
    ax.set_xlabel(f"{bands[j]} (Vega mag)"); ax.set_ylabel("Count")
    ax.set_title(f"{bands[j]} ({WISE_WAVELENGTHS_UM[bands[j]]} µm), N={det.sum():,}")

plt.tight_layout()
plt.savefig(OUT_DIR / "wise_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()


## Cell 8 — Color-color diagrams (compare to paper Figure 4 left)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

m123 = wise_data["detected"][:,0] & wise_data["detected"][:,1] & wise_data["detected"][:,2]
ax = axes[0]
if m123.sum() > 5:
    w12 = wise_data["mag"][m123,0] - wise_data["mag"][m123,1]
    w23 = wise_data["mag"][m123,1] - wise_data["mag"][m123,2]
    ax.scatter(w23, w12, s=4, alpha=0.4, c="steelblue", rasterized=True)
    ax.axhline(0.8, color="red", ls="--", lw=0.8, alpha=0.5, label='AGN line (Stern+12)')
    ax.legend(fontsize=8)
    ax.text(0.03, 0.97, f"N={m123.sum():,}", transform=ax.transAxes, ha="left", va="top",
            fontsize=9, bbox=dict(boxstyle="round", fc="wheat"))
ax.set_xlabel("W2−W3"); ax.set_ylabel("W1−W2")
ax.set_title("W1−W2 vs W2−W3 (cf. Fig 4 left)")

m1234 = m123 & wise_data["detected"][:,3]
ax = axes[1]
if m1234.sum() > 5:
    w12b = wise_data["mag"][m1234,0] - wise_data["mag"][m1234,1]
    w34  = wise_data["mag"][m1234,2] - wise_data["mag"][m1234,3]
    ax.scatter(w34, w12b, s=4, alpha=0.4, c="darkorange", rasterized=True)
    ax.text(0.03, 0.97, f"N={m1234.sum():,}", transform=ax.transAxes, ha="left", va="top",
            fontsize=9, bbox=dict(boxstyle="round", fc="wheat"))
else:
    ax.text(0.5, 0.5, f"Only {m1234.sum()} 4-band detections", transform=ax.transAxes, ha="center")
ax.set_xlabel("W3−W4"); ax.set_ylabel("W1−W2")
ax.set_title("W1−W2 vs W3−W4")

plt.tight_layout(); plt.show()

## Cell 9 — Cross-match to SDSS spectra

In [ ]:
sdss_meta_path = SDSS_DIR / "metadata.npy"
sdss_keep_idx = None

if not sdss_meta_path.exists():
    log.warning(f"{sdss_meta_path} not found. Saving all WISE rows unmatched.")
    wise_matched = wise_data
else:
    sdss_meta = np.load(sdss_meta_path)
    sdss_ids = sdss_meta["specobjid"]
    log.info(f"Loaded {len(sdss_ids):,} SDSS specobjids")
    
    wise_ids = wise_data["specobjid"]
    wise_lookup = {wid:i for i,wid in enumerate(wise_ids) if wid not in {}.__class__()}
    # (dict comp avoids dupes by keeping last, but we want first)
    wise_lookup = {}
    for i, wid in enumerate(wise_ids):
        wise_lookup.setdefault(wid, i)
    
    sdss_keep, wise_keep = [], []
    for si, sid in enumerate(sdss_ids):
        if sid in wise_lookup:
            sdss_keep.append(si)
            wise_keep.append(wise_lookup[sid])
    sdss_keep = np.array(sdss_keep, dtype=np.int64)
    wise_keep = np.array(wise_keep, dtype=np.int64)
    
    wise_matched = {k: v[wise_keep] for k,v in wise_data.items()}
    sdss_keep_idx = sdss_keep
    
    print(f"Cross-match results:")
    print(f"  SDSS spectra:       {len(sdss_ids):>8,}")
    print(f"  WISE catalogue:     {len(wise_ids):>8,}")
    print(f"  Matched (both):     {len(sdss_keep):>8,}  ({100*len(sdss_keep)/len(sdss_ids):.1f}%)")
    print(f"  SDSS-only:          {len(sdss_ids)-len(sdss_keep):>8,}")
    print(f"  WISE-only:          {len(wise_ids)-len(wise_keep):>8,}")
    
    # Verify alignment
    for i in range(min(3, len(sdss_keep))):
        ok = "✓" if sdss_ids[sdss_keep[i]] == wise_matched["specobjid"][i] else "✗"
        print(f"  Alignment check [{i}]: {ok}")

## Cell 10 — Save

In [ ]:
np.savez_compressed(OUT_DIR / "wise_photometry.npz", **wise_matched)
if sdss_keep_idx is not None:
    np.save(OUT_DIR / "wise_matched_sdss_indices.npy", sdss_keep_idx)

print(f"Saved to {OUT_DIR}/:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:<45s}  {f.stat().st_size/1024:>8.1f} kB")
print(f"\n✓ {len(wise_matched['specobjid']):,} galaxies ready")

## Cell 11 — Reload verification

In [ ]:
data = np.load(OUT_DIR / "wise_photometry.npz")
print("Reload check:")
for k in data.files:
    print(f"  {k:<15s}: shape={str(data[k].shape):<15s} dtype={data[k].dtype}")
assert data["mag"].shape[1] == 4
assert len(data["specobjid"]) == len(data["mag"])
print(f"\n✓ Integrity OK  |  z: [{data['redshift'].min():.3f}, {data['redshift'].max():.3f}]")

---
## Cell 12 — Full-scale instructions (prints runnable steps)

In [ ]:
print(f"""
╔═══════════════════════════════════════════════════════════════╗
║       FULL-SCALE: 510k galaxies                             ║
╠═══════════════════════════════════════════════════════════════╣
║                                                             ║
║ 1. Go to https://skyserver.sdss.org/casjobs/                ║
║ 2. Paste the SQL from Cell 2 (no TOP clause)                ║
║ 3. Submit → Download CSV → save to:                         ║
║    {str(CSV_PATH):<55s} ║
║ 4. In Cell 1, set:                                          ║
║      N_SPECTRA   = None                                     ║
║      DATA_SOURCE = "csv"                                    ║
║ 5. Re-run all cells.  Takes ~5s total for 510k rows.        ║
║                                                             ║
║ Memory: ~140 MB  (trivial on your 3TB server)               ║
╚═══════════════════════════════════════════════════════════════╝
""")